# 🎧 Whisper ESC‑50 Audio Classification (Google Colab)
This notebook fine‑tunes **Whisper** using **Unsloth** for environmental sound classification on the **ESC‑50 dataset**.

Run each cell in order.

In [ ]:
!uv pip install unsloth transformers datasets evaluate librosa soundfile accelerate

In [ ]:
# # ---------------------------------------------------------
# # 0. Disable TorchCodec usage and set environment
# # ---------------------------------------------------------
# import os
# import io
# os.environ["HF_DATASETS_AUDIO_ALLOW_TORCHCODEC"] = "0"

# # ---------------------------------------------------------
# # 1. Imports
# # ---------------------------------------------------------
# from datasets import load_dataset, Audio
# from unsloth import FastModel
# from transformers import WhisperProcessor, WhisperForConditionalGeneration
# import soundfile as sf
# import librosa
# import torch

# # ---------------------------------------------------------
# # 2. Load ESC-50 dataset
# # ---------------------------------------------------------
# print("Loading ESC-50 dataset...")
# # We load the dataset and immediately tell it NOT to decode audio
# ds = load_dataset("ashraq/esc50", split="train")
# ds = ds.cast_column("audio", Audio(decode=False))

# # ---------------------------------------------------------
# # 3. Load Whisper-Small from Unsloth
# # ---------------------------------------------------------
# MODEL_NAME = "unsloth/whisper-small"
# MAX_SEQ_LENGTH = 2048

# print("Loading Unsloth Whisper-Small...")
# model, tokenizer = FastModel.from_pretrained(
#     model_name=MODEL_NAME,
#     max_seq_length=MAX_SEQ_LENGTH,
#     dtype=None,
#     load_in_4bit=True,
#     auto_model=WhisperForConditionalGeneration,
#     whisper_language="English",
#     whisper_task="transcribe",
# )
# processor = WhisperProcessor.from_pretrained(MODEL_NAME)
# model.eval()

# # ---------------------------------------------------------
# # 4. Manual Audio Loading from Bytes (The "No-Path" Fix)
# # ---------------------------------------------------------
# # Get the first example
# example = ds[0]
# label = example["category"]

# print(f"\nProcessing Example Label: {label}")

# # Because decode=False, example["audio"]["bytes"] contains the raw file data (WAV/MP3)
# # We use io.BytesIO to make it look like a file for soundfile.read()
# audio_bytes = example["audio"]["bytes"]

# if audio_bytes is not None:
#     # Decode the raw bytes into a numpy array manually
#     audio_array, sr = sf.read(io.BytesIO(audio_bytes))
# else:
#     # Backup: If bytes are missing but path exists, try reading the path
#     audio_path = example["audio"].get("path")
#     if audio_path:
#         audio_array, sr = sf.read(audio_path)
#     else:
#         raise ValueError("Could not find audio bytes or a valid file path.")

# # ---------------------------------------------------------
# # 5. Resample to 16kHz (Required for Whisper)
# # ---------------------------------------------------------
# TARGET_SR = 16000
# if sr != TARGET_SR:
#     # Whisper requires 16000Hz. sf.read often returns 44100Hz for ESC-50.
#     audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR)
#     sr = TARGET_SR

# # ---------------------------------------------------------
# # 6. Process and Forward Pass
# # ---------------------------------------------------------
# # Move inputs to CUDA since Unsloth models are on GPU
# inputs = processor(
#     audio_array, 
#     sampling_rate=sr, 
#     return_tensors="pt"
# ).to("cuda")

# print("Input features shape:", inputs.input_features.shape)

# with torch.no_grad():
#     # Use generate for Whisper models to get text
#     generated_ids = model.generate(inputs.input_features)
#     transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)
    
#     print("-" * 30)
#     print(f"Target Category: {label}")
#     print(f"Whisper Transcription: {transcription[0]}")
#     print("-" * 30)

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import json
from pathlib import Path

from unsloth import FastModel
from transformers import WhisperForConditionalGeneration
from datasets import load_dataset, ClassLabel, Audio
from transformers import WhisperProcessor, TrainingArguments, Trainer
import evaluate
import librosa
import soundfile as sf

# ============================================
# CONFIGURATION
# ============================================
MODEL_NAME = "unsloth/whisper-small"
NUM_CLASSES = 50
OUTPUT_DIR = "./whisper_esc50_finetuned"

MAX_LENGTH = 128
BATCH_SIZE = 1
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LENGTH = 2048

In [ ]:
def load_and_prepare_model():
    print(f"Loading model: {MODEL_NAME}")

    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        auto_model=WhisperForConditionalGeneration,
        whisper_language="English",
        whisper_task="transcribe",
    )

    hidden_size = model.config.hidden_size
    print(f"Replacing lm_head with classification head: {hidden_size} -> {NUM_CLASSES}")

    model_config = {
        "original_hidden_size": hidden_size,
        "num_classes": NUM_CLASSES,
        "model_name": MODEL_NAME
    }

    model.lm_head = nn.Linear(hidden_size, NUM_CLASSES)

    model = FastModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    return model, tokenizer, model_config

In [ ]:
def load_esc50_dataset():
    print("Loading ESC-50 dataset...")

    dataset = load_dataset("ashraq/esc50", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))
    dataset = dataset.add_column("label", [i for i in dataset["category"]])
    dataset = dataset.add_column("class_name", [i for i in dataset["label"]])
    dataset = dataset.cast_column("label", ClassLabel(names=list(set(dataset["label"]))))

    dataset = dataset.train_test_split(
        test_size=0.2, seed=42, stratify_by_column="label"
    )

    print(f"Training samples: {len(dataset['train'])}")
    print(f"Validation samples: {len(dataset['test'])}")

    return dataset

In [ ]:
def preprocess_audio(example, processor):
    audio = example["audio"]

    inputs = processor(
        audio["array"],
        sampling_rate=16000,
        return_tensors="pt"
    )

    example["input_features"] = inputs["input_features"].squeeze(0)
    example["labels"] = example["label"]

    return example

import io
import soundfile as sf
import librosa

def preprocess_audio(example, processor):
    # 1. Access the audio dict (which has decode=False)
    audio_info = example["audio"]
    audio_bytes = audio_info.get("bytes")
    
    # 2. Decode the raw bytes manually
    if audio_bytes is not None:
        # Wrap bytes in BytesIO so soundfile can read it like a file
        audio_array, sr = sf.read(io.BytesIO(audio_bytes))
    else:
        # Fallback to path if bytes are missing (unlikely in ESC-50)
        audio_path = audio_info.get("path")
        if audio_path:
            audio_array, sr = sf.read(audio_path)
        else:
            raise ValueError("No audio bytes or path found in example.")

    # 3. Handle multi-channel (Stereo to Mono)
    if len(audio_array.shape) > 1:
        audio_array = audio_array.mean(axis=-1)

    # 4. Resample to 16kHz (Whisper Requirement)
    TARGET_SR = 16000
    if sr != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR

    # 5. Process through WhisperProcessor
    inputs = processor(
        audio_array, 
        sampling_rate=sr, 
        return_tensors="pt"
    )

    # 6. Prepare the final dictionary for the Trainer
    return {
        "input_features": inputs.input_features[0], # Remove batch dim
        "labels": example["label"]                  # Ensure this matches your column name
    }

In [ ]:
class AudioClassificationDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        # Convert each item to a tensor if it isn't one already
        input_features = torch.stack([
            torch.tensor(item["input_features"]) if not isinstance(item["input_features"], torch.Tensor)
            else item["input_features"]
            for item in batch
        ])

        labels = torch.tensor([item["labels"] for item in batch])

        return {
            "input_features": input_features,
            "labels": labels
        }

In [ ]:
class WhisperClassificationTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.eval_metric = evaluate.load("accuracy")

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        input_features = inputs["input_features"]
        labels = inputs["labels"]

        outputs = model.base_model.model.model.encoder(input_features)
        pooled_output = outputs.last_hidden_state.mean(dim=1)
        logits = model.lm_head(pooled_output)

        loss = nn.CrossEntropyLoss()(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        input_features = inputs["input_features"]
        labels = inputs["labels"]

        outputs = model.base_model.model.model.encoder(input_features)
        pooled_output = outputs.last_hidden_state.mean(dim=1)
        logits = model.lm_head(pooled_output)

        loss = nn.CrossEntropyLoss()(logits, labels)
        return (loss, logits, labels)

    def compute_metrics(self, eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        accuracy = self.eval_metric.compute(predictions=predictions, references=labels)["accuracy"]
        return {"accuracy": accuracy}

In [ ]:
def save_model_with_adapter(model, tokenizer, model_config, save_path):
    print(f"Saving model to {save_path}...")
    os.makedirs(save_path, exist_ok=True)

    with open(os.path.join(save_path, "model_config.json"), "w") as f:
        json.dump(model_config, f, indent=2)

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    torch.save(model.lm_head.state_dict(), os.path.join(save_path, "classification_head.pth"))
    print("✓ Model saved.")


def load_model_with_adapter(model_path, device="cuda"):
    print(f"Loading model from {model_path}...")

    with open(os.path.join(model_path, "model_config.json"), "r") as f:
        model_config = json.load(f)

    model, tokenizer = FastModel.from_pretrained(
        model_name=model_config["model_name"],
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        auto_model=WhisperForConditionalGeneration,
        whisper_language="English",
        whisper_task="transcribe",
    )

    hidden_size = model_config["original_hidden_size"]
    num_classes = model_config["num_classes"]
    model.lm_head = nn.Linear(hidden_size, num_classes)

    model.load_adapter(model_path, "default")

    head_path = os.path.join(model_path, "classification_head.pth")
    model.lm_head.load_state_dict(torch.load(head_path, map_location=device))

    model = model.to(device)
    print("✓ Model loaded successfully.")
    return model, tokenizer, model_config

In [ ]:
def train_model():
    model, tokenizer, model_config = load_and_prepare_model()
    dataset = load_esc50_dataset()
    processor = WhisperProcessor.from_pretrained(MODEL_NAME)

    print("Preprocessing audio...")
    dataset["train"] = dataset["train"].map(
        lambda x: preprocess_audio(x, processor),
        remove_columns=["audio", "category", "fold"]
    )
    dataset["test"] = dataset["test"].map(
        lambda x: preprocess_audio(x, processor),
        remove_columns=["audio", "category", "fold"]
    )

    data_collator = AudioClassificationDataCollator(tokenizer)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none",
    )

    trainer = WhisperClassificationTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["train"],
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()
    print("Evaluating...")
    print(trainer.evaluate())

    save_model_with_adapter(model, tokenizer, model_config, OUTPUT_DIR)
    return trainer, dataset

In [ ]:
# Run training
trainer, dataset = train_model()

In [ ]:
def classify_audio(model, processor, audio_path):
    model.eval()
    audio, sr = librosa.load(audio_path, sr=16000)

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs["input_features"].squeeze(0)

    with torch.no_grad():
        outputs = model.base_model.model.model.encoder(input_features)
        pooled_output = outputs.last_hidden_state.mean(dim=1)
        logits = model.lm_head(pooled_output)
        probabilities = torch.softmax(logits, dim=-1)

    predicted_class = torch.argmax(probabilities).item()
    confidence = probabilities[0, predicted_class].item()

    return predicted_class, confidence

In [ ]:
# Load saved model
model, tokenizer, model_config = load_model_with_adapter(OUTPUT_DIR)
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

# Example usage:
# Upload a file in Colab and set its path here
# pred, conf = classify_audio(model, processor, "/content/test.wav")
# print(pred, conf)